# Penggabungan Data

Membaca 8 file bersih dari `data/processed/` dan menggabungkannya menjadi satu tabel harian: `data/processed/data_gabungan.csv`.

1. Semua indikator di-reindex ke kalender hari kerja harga emas (join **by date**).
2. `bi_rate` (jarang berubah) di-forward-fill. `ffr` adalah FEDFUNDS **bulanan** (rata-rata sebulan, bertanggal tgl-1): rata-rata bulan M baru diketahui setelah bulan M berakhir, jadi dipakai mulai bulan **M+1** dan dipetakan **as-of** (bukan reindex tanggal persis), sehingga bulan yang tgl-1-nya jatuh di akhir pekan/libur tidak terbuang. (Protokol v9: sebelumnya nilai bulan M dipakai sejak tgl-1 bulan M = lookahead, dan 29% bulan terbuang.)
3. Sisa NaN (selisih hari libur antar negara) diisi forward-fill lalu backward-fill.
4. **Seluruh kolom eksogen (7 indikator) di-shift 1 hari kerja** — nilai indikator hari *t-1* dipakai untuk memprediksi target hari *t*, supaya tidak ada kebocoran informasi masa depan (*lookahead bias*).
5. Target `emas_ret` = return harian harga emas (%), dihitung **sebelum** kolom lain di-lag.
6. **Fitur eksogen stasioner**: selain level mentah, dibuat 7 fitur `*_ret` (log-return: DXY, S&P 500, USD/IDR, IHSG) dan `*_diff` (selisih harian: US10YR, FFR, BI-Rate). **Model hanya memakai fitur stasioner ini**; level mentah disimpan untuk EDA/ADF saja. Alasannya (notebook 03): level S&P 500, IHSG, dan FFR di periode test berada di luar rentang train hampir sepanjang waktu, sehingga model terpaksa mengekstrapolasi. Fitur stasioner dihitung dari level yang sama lalu di-shift 1 hari (`diff(shift(x)) == shift(diff(x))`), jadi tidak ada lookahead.

In [11]:
import sys
import pandas as pd
import numpy as np
import os

sys.path.insert(0, os.path.join('..'))
from src.pipeline import make_exo_features, EXO_LEVEL_COLS, EXO_COLS

PROCESSED = os.path.join('..', 'data', 'processed')

## 1. Muat 8 file bersih

In [12]:
harga_jual = pd.read_csv(os.path.join(PROCESSED, 'harga_emas_bersih.csv'), index_col=0, parse_dates=True)['harga_jual']

nama_indikator = ['bi_rate', 'dxy', 'sp500', 'usd_idr', 'ihsg', 'us10yr', 'ffr']
sumber = {
    nama: pd.read_csv(os.path.join(PROCESSED, f'{nama}_bersih.csv'), index_col=0, parse_dates=True).iloc[:, 0]
    for nama in nama_indikator
}

print(f'harga_jual: {len(harga_jual)} baris')
for nama, s in sumber.items():
    print(f'{nama:10s}: {len(s)} baris')


harga_jual: 4326 baris
bi_rate   : 5175 baris
dxy       : 4340 baris
sp500     : 4339 baris
usd_idr   : 4466 baris
ihsg      : 4180 baris
us10yr    : 4346 baris
ffr       : 208 baris


## 2. Reindex ke kalender hari kerja emas & imputasi

In [13]:
df = pd.DataFrame({'harga_jual': harga_jual})
for nama, s in sumber.items():
    if nama == 'ffr':
        continue                       # ditangani khusus di bawah (bulanan, as-of, lag 1 bulan)
    df[nama] = s.reindex(df.index)

# FFR = FEDFUNDS BULANAN. Rata-rata bulan M baru diketahui setelah bulan M berakhir ->
# dipakai mulai bulan M+1, dipetakan 'as-of' (ffill) supaya tgl-1 akhir pekan tidak terbuang.
ffr_tersedia = sumber['ffr'].copy()
ffr_tersedia.index = ffr_tersedia.index + pd.offsets.MonthBegin(1)
df['ffr'] = ffr_tersedia.reindex(df.index.union(ffr_tersedia.index)).ffill().reindex(df.index)
print(f"FFR: {len(ffr_tersedia)} bulan; perubahan nilai terlihat di data harian: "
      f"{int((df['ffr'].diff().fillna(0) != 0).sum())}")

print('NaN per kolom sebelum imputasi:')
print(df.isna().sum())

# frekuensi rendah -> forward-fill (nilai tetap berlaku sampai ada perubahan baru)
for c in ['ffr', 'bi_rate']:
    df[c] = df[c].ffill()

# sisa NaN (selisih kalender libur antar negara) -> forward-fill lalu backward-fill
df = df.ffill().bfill()
print('\nNaN residual sesudah imputasi:', int(df.isna().sum().sum()))


FFR: 208 bulan; perubahan nilai terlihat di data harian: 132
NaN per kolom sebelum imputasi:
harga_jual      0
bi_rate       282
dxy           155
sp500         156
usd_idr        34
ihsg          305
us10yr        178
ffr             0
dtype: int64

NaN residual sesudah imputasi: 0


## 3. Cegah Lookahead Bias, Hitung Target & Fitur Stasioner

Tujuh kolom eksogen di-*shift* satu hari kerja sebelum digabung dengan harga. Target (`emas_ret`) dihitung dari harga **sebelum** di-lag, karena ia bukan bagian dari fitur eksogen. Fitur stasioner (`dxy_ret`, ..., `bi_rate_diff`) dihitung dari level yang sama lewat `make_exo_features` (di `src/pipeline.py`, satu-satunya definisi) lalu di-shift dengan cara yang sama.

In [14]:
exo_cols = [c for c in df.columns if c != 'harga_jual']          # 7 level mentah
assert set(exo_cols) == set(EXO_LEVEL_COLS), exo_cols
df_exo_lag = df[exo_cols].shift(1)                               # level, lag-1 (untuk EDA/ADF)

# fitur STASIONER yang dipakai model: log-return / selisih, lalu lag-1 (tanpa lookahead)
df_exo_stat = make_exo_features(df[EXO_LEVEL_COLS]).shift(1)

df['emas_ret'] = df['harga_jual'].pct_change() * 100

gabungan = pd.concat([df[['harga_jual', 'emas_ret']], df_exo_lag, df_exo_stat], axis=1)
gabungan = gabungan.dropna()
assert np.isfinite(gabungan.values).all(), 'ada NaN/inf di data gabungan'
assert all(c in gabungan.columns for c in EXO_COLS)

print(f'Data gabungan akhir: {gabungan.shape[0]} baris x {gabungan.shape[1]} kolom')
print(f'Periode: {gabungan.index.min().date()} s.d. {gabungan.index.max().date()}')
print('Kolom level   :', exo_cols)
print('Kolom stasioner (dipakai model):', EXO_COLS)
gabungan.head()

Data gabungan akhir: 4325 baris x 16 kolom
Periode: 2010-01-05 s.d. 2026-08-03
Kolom level   : ['bi_rate', 'dxy', 'sp500', 'usd_idr', 'ihsg', 'us10yr', 'ffr']
Kolom stasioner (dipakai model): ['dxy_ret', 'sp500_ret', 'usd_idr_ret', 'ihsg_ret', 'us10yr_diff', 'ffr_diff', 'bi_rate_diff']


,harga_jual,emas_ret,bi_rate,dxy,sp500,usd_idr,ihsg,us10yr,ffr,dxy_ret,sp500_ret,usd_idr_ret,ihsg_ret,us10yr_diff,ffr_diff,bi_rate_diff
tanggal,,,,,,,,,,,,,,,,
2010-01-05,410000.0,0.490196,6.5,77.529999,1132.989990,9364.200195,2575.312988,3.85,0.12,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0
2010-01-06,410000.0,0.000000,6.5,77.620003,1136.520020,9305.200195,2605.175537,3.77,0.12,0.001160,0.003111,-0.006321,0.011529,-0.08,0.0,0.0
2010-01-07,412000.0,0.487805,6.5,77.489998,1137.140015,9218.099609,2603.195557,3.85,0.12,-0.001676,0.000545,-0.009405,-0.000760,0.08,0.0,0.0
2010-01-08,410000.0,-0.485437,6.5,77.910004,1141.689941,9272.500000,2586.794189,3.85,0.12,0.005405,0.003993,0.005884,-0.006320,0.00,0.0,0.0
2010-01-11,413000.0,0.731707,6.5,77.470001,1144.979980,9212.700195,2614.268311,3.83,0.12,-0.005664,0.002878,-0.006470,0.010565,-0.02,0.0,0.0


## 4. Simpan

In [15]:
out_path = os.path.join(PROCESSED, 'data_gabungan.csv')
gabungan.to_csv(out_path)
print(f'Tersimpan: {out_path}')
gabungan.describe().T[['mean', 'std', 'min', 'max']].round(3)


Tersimpan: ..\data\processed\data_gabungan.csv


,mean,std,min,max
harga_jual,886115.260,534841.960,394000.000,3168000.000
emas_ret,0.046,0.754,-6.046,5.495
bi_rate,5.646,1.204,3.500,7.750
dxy,92.971,9.281,72.930,114.110
sp500,3093.306,1655.368,1022.580,7609.780
usd_idr,13213.850,2498.663,8292.700,18190.301
ihsg,5586.806,1326.577,2475.476,9134.700
us10yr,2.673,1.036,0.520,4.980
ffr,1.477,1.821,0.050,5.330
dxy_ret,0.000,0.004,-0.024,0.020
